# S08 · Debug portability and application boundaries

**Outcome:** Separate syntax, data, semantics and implementation issues when a query behaves unexpectedly.

**Time:** about 45 minutes. Run cells in order. This is the executed solution edition.

Debug a query in layers. First parse it. Next confirm the dataset and graph. Inspect IRIs, language tags and datatypes. Count intermediate solution rows before adding OPTIONAL, aggregation or paths. Then check the configured entailment regime and engine-specific extensions. Compare actual bindings, not only row counts.

The pwin course's engine comparisons are useful evidence of how portability problems arise; they are not a guarantee that this package has tested the same engines. This package's execution report names its own tested environment. RDF 1.2, SPARQL 1.2 and GeoSPARQL are extension study topics here, outside the SPARQL 1.1 core execution contract. Do not feed unsupported triple-term syntax or extension functions to a DL parser and claim success.

Federation needs operational controls: an allowlist of destinations, timeouts, a clear policy on sending bound values, stable remote term identities and visible partial failures. SERVICE SILENT can preserve rows after a remote failure but can conceal incomplete enrichment. The course keeps network federation optional so credentials and endpoint availability do not block learning. The HTTP helper separates query, update and graph responses and raises failures instead of presenting them as empty answers.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Parse standard syntax and detect a nonstandard form

In [2]:
from rdflib.plugins.sparql.parser import parseQuery
good=PREFIX+'SELECT ?r ?band WHERE { ?r ex:stayDays ?d BIND(IF(?d >= 7,"longer","shorter") AS ?band) }'
parseQuery(good)
rejected=False
try:parseQuery('SELECT ?x WHERE { LET (?x := 1) }')
except Exception:rejected=True
assert rejected
print('Portable query parsed; nonstandard LET rejected.')

Portable query parsed; nonstandard LET rejected.


## Compare equivalent answer sets

In [3]:
g=build_asserted()
q1='SELECT ?r WHERE { ?r ex:primaryCode <https://example.org/health/icd9-source/493> }'
q2='SELECT ?r WHERE { ?r ex:primaryCode ?c VALUES ?c { <https://example.org/health/icd9-source/493> } }'
a={str(r[0]) for r in query(g,q1)};b={str(r[0]) for r in query(g,q2)}
assert a==b and len(a)==20
print('Equivalent named answers:',len(a))

Equivalent named answers: 20


## Your turn

Return a debugging sequence list containing syntax, dataset, terms, joins and entailment in that order.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = ['syntax','dataset','terms','joins','entailment']

In [5]:
learner_check(answer, lambda x:x==['syntax','dataset','terms','joins','entailment'], 'Start with checks that can be resolved locally.')

Exercise passed.
Out[0]: True


## Explain your model

Why is changing an HTTP failure into an empty result dangerous for a data product?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.